In [5]:
import sys
print(sys.prefix)

C:\Users\mosab\anaconda3\envs\tf


In [6]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Check if a GPU is available and set the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bigscience/bloomz-7b1")

# Load the model and quantize it to 4-bit
model = AutoModelForCausalLM.from_pretrained("bigscience/bloomz-7b1", 
                                             #device_map='auto', 
                                             load_in_8bit=True,
                                             torch_dtype=torch.float16,
                                             #bnb_4bit_compute_dtype=torch.float16  # Set the compute data type to float16 for better performance
                                            )#.to(device)

# Load the ARCD dataset from CSV
df = pd.read_csv(r'C:\Users\mosab\Desktop\Datasets\General\ARCD\arcd_all.csv', encoding='utf-16', sep='\t')

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


In [ ]:
# import pandas as pd
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
# from tensorflow import keras

# model_path = r'C:\Users\mosab\Desktop\Jupyter Notebook\Research\models\'
# # Load the tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_path + "bloomz-7b1 - tokenizer - standard training")
# loaded_model = keras.models.load_model(model_path + "bloomz-7b1 - model - standard training")

# # Load the ARCD dataset from CSV
# df = pd.read_csv(r'C:\Users\mosab\Desktop\Datasets\General\ARCD\arcd_all.csv', encoding='utf-16', sep='\t')

In [ ]:
df.shape

In [7]:
import numpy as np
import re

#df=df1[i:i+50]
#i+=50
# Function to extract answers using BLOOMZ-7B1
def extract_answers_from_context(question, context, title):
    context=f"Title: {title}\n"+context
    # Combine title, context, and question to create the prompt
    prompt = f"Context: {context}\nQuestion: {question}\nInstruction: When there is no answer in the context, don't output any word.\nAnswer:"
    
    # Tokenize input and generate output using the model
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(inputs['input_ids'], max_new_tokens=300, do_sample=False, pad_token_id=tokenizer.eos_token_id)#max_length=356
    
    # Decode the generated output
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract the answer from the generated text
    answer = generated_text.split("Answer:")[-1].strip()
    return answer

# To match only Arabic letters in Regex, we use unicode:

# [\u0621-\u064A]+
# Or we can simply use Arabic letters directly

# [ء-ي]+

# Text normalization for comparison (lowercase, remove articles, punctuation, etc.)
def normalize_text(text):
    text = text.lower()
    #text = re.sub(r'\b(a|an|the)\b', ' ', text)  # Remove articles
    text = re.sub(r'[\u064B-\u065F]', '', text)  # Remove diacritics
    text = re.sub(r'\W+', ' ', text)  # Remove punctuation and double whitespaces
    #text = re.sub(r'[^a-zء-ي0-9]+', ' ', text)  # Remove double whitespaces
    #text = ' '.join(text.split())  # Remove extra spaces
    return text

# Exact Match (EM) calculation
def exact_match_score(predicted_answer, actual_answer):
    return int(normalize_text(predicted_answer) == normalize_text(actual_answer))

# F1 Score calculation for a single prediction
def f1_single(predicted_answer, actual_answer):
    # Tokenize predicted and actual answers
    pred_tokens = normalize_text(predicted_answer).split()
    actual_tokens = normalize_text(actual_answer).split()
    
    if (len(pred_tokens) == 0) and (len(actual_tokens) == 0):
        return 1
    elif (len(pred_tokens) == 0):
        return 0
    elif (len(actual_tokens) == 0):
        return 0
     #common_tokens = pred_tokens.intersection(actual_tokens)
    common_tokens = set(pred_tokens) & set(actual_tokens)
    #common_tokens = remove_stopwords_from_set(common_tokens, stopwords_list)
    if len(common_tokens) == 0:
        return 0.0
    
    precision = len(common_tokens) / len(pred_tokens)
    recall = len(common_tokens) / len(actual_tokens)
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

# Function to calculate average EM and F1 scores across the dataset
def evaluate_qa(predicted_answers, actual_answers):
    total_em = 0
    total_f1 = 0
    n = len(predicted_answers)
    
    for predicted, actual in zip(predicted_answers, actual_answers):
        total_em += exact_match_score(predicted, actual)
        total_f1 += f1_single(predicted, actual)
    
    # Calculate the average EM and F1 scores
    em_score = total_em / n
    f1_score_avg = total_f1 / n
    return em_score, f1_score_avg

In [ ]:
questions=[]
contexts=[]
# List to store predicted and actual answers
predicted_answers = []
actual_answers = []

n = len(df)
# Iterate over the dataset and extract answers for each row
for index, row in tqdm(df.iterrows(), total=n):
    question = row["question"]
    context = row["context"]
    title = row["title"]
    
    questions.append(question)
    contexts.append(context)
    # Extract answer from the model
    predicted_answer = extract_answers_from_context(question, context, title)
    predicted_answers.append(predicted_answer)
    
    # Get the actual answer(s) from the dataset
    actual_answer = row["answer"]
    actual_answers.append(actual_answer)

# Calculate Exact Match and F1 Score
em_score, f1_score_avg = evaluate_qa(predicted_answers, actual_answers)
print(f"\nExact Match Score (EM): {em_score:.2f}")
print(f"F1 Score: {f1_score_avg:.2f}")

In [6]:
total=list(zip(contexts,questions, actual_answers, predicted_answers))
length=len(total)
j=(int)(length/10)
slices=[]
for i in range(10):
    slices += total[i*j:i*j+10]

for context, question, actual_answer, predicted_answer in slices:
    print(context)
    print(question)
    print(actual_answer)
    print(predicted_answer)
    print("=============================================")
i+=30  

NameError: name 'contexts' is not defined